In [ ]:
import numpy as np
import pandas as pd


#导包


In [ ]:
movie_df = pd.read_csv('movie.csv', encoding='utf-8')
movie_df.head()
#导入数据

In [ ]:
#检查缺失值
print(movie_df.isnull())
movie_df.info

In [ ]:
#处理缺失值
for column in movie_df.columns:
    if np.all(pd.notnull(movie_df[column])) == False:
        movie_df.dropna(axis=0, inplace=True)
movie_df.info


In [ ]:
rate_counts = movie_df['Rating'].value_counts().sort_index()
rate_counts
#评分分布

In [ ]:
#评分最高的20部
movie_best = movie_df.sort_values('Rating', ascending=False)
movie_best[:20]['Title']

In [ ]:
#先提取所有类型
genre_series = movie_df.Genre
dict = {}
for genre in genre_series:
    genre_list = genre.strip().split(',')
    for i in genre_list:
        if i not in dict.keys():
            dict[i] = 0
dict_sort = sorted(dict.items(), key=lambda x: x[0], reverse=False)

dict1 = {}

for t in dict_sort:
    dict1[t[0]] = t[1]
print(dict1)
print(len(dict1))
dict2 = dict1




In [ ]:
#再进行计数
for genre in genre_series:
    genre_list = genre.strip().split(',')
    for ele in genre_list:
        if ele in dict2.keys():
            dict2[ele] += 1

data_df = pd.DataFrame(
    list(dict2.items()),  # 使用items()确保顺序一致
    columns=['genre', 'count']
)

# 按count降序排序，并重置索引
data_df = data_df.sort_values(by='count', ascending=False).reset_index(drop=True)

print("电影类型统计:")
print(data_df)







In [ ]:
#电影类型统计的向量化
movie_df_copy = movie_df.copy()
movie_df_copy['Genre_list'] = movie_df_copy['Genre'].str.split(',')
movie_df_copy = movie_df_copy.explode('Genre_list')
movie_df_copy_1 = movie_df_copy.groupby('Genre_list').agg({'Genre_list': 'count'})
movie_df_copy_1




In [ ]:
#各类型的平均评分与排名
data1 = movie_df.groupby(['Genre']).agg({'Rating': 'mean', 'Rank': 'mean'})
data1 = data1[['Rating', 'Rank']].round(1)
data1.sort_values(by=['Rating', 'Rank'], ascending=[False, False], inplace=True)
data1

In [ ]:
# 类型流行趋势
data2 = movie_df[['Year', 'Genre']]
data2.groupby(['Year']).agg({'Genre': 'count'})  #每年的类型数量
data2['Genre_list'] = data2['Genre'].str.split(',')
# print(data2)
data2 = data2.explode('Genre_list')  #explode方法是将一个单元格中的列表，每个元素都拿出来，形成一列
# print(data2)
series2 = data2.groupby(['Year', 'Genre_list']).size()
print(series2.index)
series1 = data2.groupby(['Year', 'Genre_list']).size().reset_index(name='count')  #- 功能 ：计算每个分组中的元素数量
# - 输出 ：返回一个Series，索引是分组键(年份和类型)，值是该分组的大小

#reset_index(name=):给series的值列重命名
data_series1 = pd.DataFrame(series1)
data_series1






In [ ]:
#不同类型的电影的评分分布对比
data3 = movie_df[['Genre', 'Rating']]
data3['Genre_list'] = data3['Genre'].str.split(',')
date3_exploded = data3.explode('Genre_list')
data3_1 = date3_exploded.groupby(['Genre_list']).agg({'Rating': 'mean'}).round(2).sort_values(by=['Rating'],
                                                                                              ascending=[False])
data3_1


In [ ]:
#电影产量随时间变化（按年份）
data4 = movie_df[['Title', 'Year']]
data4_1 = data4.groupby(['Year']).agg(Count=('Title', 'count'))
data4_1

In [ ]:
#电影产量随时间变化（按年代）
data4['Decade'] = (data4['Year'] // 10 * 10).astype(str)  #添加一列年代
decade_counts = data4.groupby(['Decade']).agg({'Title': 'count'}).reset_index()
decade_counts

In [ ]:
#不同年代的电影的平均评分对比
data5 = movie_df[['Rating', 'Year']]
data5_1 = data5.groupby('Year').agg({'Rating': 'mean'}).round(2).sort_values(by='Rating', ascending=False)
# print(data5_1)

data5['Decade'] = (data5['Year'] // 10 * 10).astype(str)  #添加一列年代
data5_2 = data5.groupby(['Decade']).agg({'Rating': 'mean'})
data5_2

In [ ]:
#电影时长随时间的变化趋势
data6 = movie_df[['Year', 'Runtime (Minutes)']]
data6['Decade'] = (data6['Year'] // 10 * 10).astype(str)  #添加一列年代

data6_1 = data6.groupby(['Decade']).agg({'Runtime (Minutes)': 'mean'}).round(2)
data6_1

In [ ]:
#最佳的电影时长是多少（评分最高的时长区间）
data7 = movie_df[['Rating', 'Runtime (Minutes)']]
data7['Runtime (Minutes)'] = data7['Runtime (Minutes)'] / 60
data7.rename({'Runtime (Minutes)': 'Runtime (Hours)'}, axis=1, inplace=True)
data7['Runtime (Hours)'].max()
bins = [0, 1.5, 2, 2.5, 3]  #每两个值之间为一个区间

data7['Runtime_Interval'] = pd.cut(
    data7['Runtime (Hours)'],  #需要分类的数据
    bins=bins,
    labels=['短', '中', '长', '超长'],  #每一列的标签

)

runtime_counts = data7["Runtime_Interval"].value_counts().sort_index()
runtime_counts_df = pd.DataFrame(data=runtime_counts).reset_index(drop=False)
runtime_counts_df


In [ ]:
#不同类型电影的时长的差异
data8 = movie_df[['Genre', 'Runtime (Minutes)']].copy()
data8['Genre_list'] = data8['Genre'].str.split(',')
data8_exploded = data8.explode('Genre_list')
data8_1 = data8_exploded.groupby(['Genre_list']).agg({'Runtime (Minutes)': 'mean'}).round(2).sort_values(
    by=['Runtime (Minutes)'], ascending=[False])
data8_1


In [ ]:
#高产导演top20

data9 = movie_df[['Director', 'Title']].copy()
data9_1 = data9.groupby(['Director']).agg({'Title': 'count'}).reset_index().sort_values(by=['Title'], ascending=[False])
data9_1.head(20)

In [ ]:
#高评分导演Top20
data10 = movie_df[['Director', 'Title', 'Rating']].copy()
data10_1 = data10.groupby(['Director']).agg({'Rating': 'mean', 'Title': 'count'}).reset_index().sort_values(
    by=['Rating'], ascending=[False])
data10_1[data10_1['Title'] >= 4][:20]['Director']


In [ ]:
#导演的票房表现
data11 = movie_df[['Director', 'Revenue (Millions)']].copy()
#缺失值处理
data11.dropna(inplace=True)
#分析
data11_1 = data11.groupby(['Director']).agg({'Revenue (Millions)': 'sum'}).reset_index().sort_values(
    by=['Revenue (Millions)'], ascending=[False])
data11_1

In [ ]:
#导演类型偏好分析
data12 = movie_df[['Director', 'Genre']].copy()
data12['Genre_list'] = data12['Genre'].str.split(',')
data12_exploded = data12.explode('Genre_list')
data12_1 = data12_exploded.groupby(['Director', 'Genre_list']).size().reset_index(name='count')

# 找出每个导演计数最高的类型
top_genres_by_director = data12_1.groupby('Director').apply(
    lambda x: x.nlargest(1, 'count')  # nlargest:前n大 的值, x指的是每个分类的dataframe
).reset_index(drop=True)

# 按计数排序
top_genres_by_director = top_genres_by_director.sort_values('count', ascending=False)

print("每个导演最擅长的类型 (按计数排序):")
top_genres_by_director.head(20)


In [ ]:
#出演电影最多的演员top20
data13 = movie_df[['Actors', 'Title']].copy()
data13['Actor_list'] = data13['Actors'].str.split(',')
data13_exploded = data13.explode('Actor_list')

data13_1 = data13_exploded.groupby(['Actor_list']).agg({'Title': 'count'}).sort_values(by=['Title'], ascending=[False])
data13_1.head(20)

In [ ]:
#电影评分最高的演员（主演）
data14 = movie_df[['Actors', 'Rating']].copy()
main_actor = []
for i in range(data14.shape[0]):
    name_list = data14['Actors'].iloc[i].split(',')  #手搓  iloc[]是按排位取值 []是取索引中的值
    main_actor.append(name_list[0])
name_series = pd.Series(main_actor)
name_series.dropna(inplace=True)
data14['Actor_main'] = name_series
data14_1 = data14.groupby(['Actor_main']).agg({'Rating': 'mean'}).reset_index().sort_values(by=['Rating'],
                                                                                            ascending=[False])
data14_1.nlargest(1, 'Rating', keep='all')




In [ ]:
#演员类型偏好
data15 = movie_df[['Actors', 'Genre']].copy()
data15_clean = data15.dropna()
data15_clean['Genre'] = data15_clean['Genre'].str.split(',')
data15_clean = data15_clean.explode('Genre')
data15_clean['Actor_main'] = data15_clean['Actors'].str.split(',').str[0]
# data15_clean
data15_1 = data15_clean.groupby(['Actor_main', 'Genre']).size().reset_index(name='count')
data15_2 = data15_1.groupby(['Actor_main']).apply(lambda x: x.nlargest(1, 'count', keep='first')).reset_index(drop=True)
data15_2
# for i in range(0, data15.shape[0]):
#     name_list = data15['Actors'].iloc[i].split(',')  #手搓
#     main_actor.append(name_list[0])
# name_series = pd.Series(main_actor)
# name_series.dropna()
# data15['Actor_main'] = name_series
# data15['Genre_list'] = data15['Genre'].str.split(',')
# data15_exploded = data15.explode('Genre_list')
# data15_1 = data15_exploded.groupby(['Actor_main', 'Genre_list']).size().reset_index(name='count')
#
# top_genres_by_actor = data15_1.groupby(['Actor_main']).apply(lambda x: x.nlargest(1, 'count')).reset_index(drop=True)
# top_genres_by_actor



In [268]:
"""
后续
基于用户标签或评论：
1. 分析用户对电影的情感倾向
2. 挖掘高频关键词
3. 建立简单的情感分类模型
"""
"""
如果有票房数据：
1. 分析影响票房的因素（评分、类型、导演、时长等）
2. 建立简单的线性回归模型
3. 预测电影是否成功（评分>4或票房高于平均水平）
"""

'\n基于用户标签或评论：\n1. 分析用户对电影的情感倾向\n2. 挖掘高频关键词\n3. 建立简单的情感分类模型\n'